# Preparacion reproducible en Colab
Seleccionar runtime GPU T4 para entrenamiento/inferencia. Repositorio privado: cargar un ZIP autorizado mediante selector; no escribir tokens en celdas. Se trabaja desde raiz del proyecto. Las salidas vacias no constituyen evidencia de ejecucion.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    z.extractall('/content/proyecto')
roots = list(Path('/content/proyecto').rglob('requirements.txt'))
assert len(roots) == 1, 'El ZIP debe contener una sola copia del proyecto'
os.chdir(roots[0].parent)
print(Path.cwd())

In [ ]:
%pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
%pip install -r requirements.txt
!python -m pip check

In [ ]:
import torch, transformers, peft, accelerate, bitsandbytes, datasets
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
print('transformers', transformers.__version__, 'PEFT', peft.__version__)
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## MySQL real requerido
Configurar un servidor MySQL **8.4.10** accesible desde Colab con ONLY_FULL_GROUP_BY. Docker Compose del repositorio puede ejecutarse en un host con Docker; Colab no incluye daemon Docker. Usar usuarios distintos para preparacion y lectura. Introducir credenciales mediante getpass y variables de entorno, nunca guardarlas en salidas. Las propuestas no cuentan como aceptadas hasta validacion real y revision semantica.

In [ ]:
!python -m scripts.bootstrap_mysql_colab
from scripts.with_env import load_env
load_env()


In [ ]:
from src.data.generate import generate
print(generate())

In [ ]:
!python -m scripts.with_env python -m scripts.build_dataset initialize
!python -m scripts.with_env python -m scripts.build_dataset validate
!python -m scripts.select_validation
